# Data scarcity baselines: 7 labelled WISDM users

Classical rows of Table 2. Same users and test set as `data_scarcity.ipynb`; features: 40 statistical, 33 temporal or the 3 FIN features per axis.

In [5]:
import os
import sys

FIN = os.path.dirname(os.getcwd())
sys.path[:0] = [FIN, os.path.join(FIN, "data-processing")]

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import config as cfg
import statistical_features as ignatov
import wisdm_windowing as ww

SEEDS = (0, 1, 2, 3, 4)
N_TRAIN = 7
N_VAL = 2

## Data and features

In [6]:
windows, users, labels = ww.window_raw()
candidates = np.setdiff1d(np.unique(users), [30])

w = windows.astype(np.float64)
axes = [ww.temporal_axis(w[:, a]) for a in range(3)]
FEATURES = {
    "40 statistical": np.stack([ignatov.extract_basic_features(*x) for x in w]),
    "33 temporal": np.stack([axes[a][f] for f in cfg.TEMPORAL for a in range(3)], axis=1),
    "3 FIN features": np.stack([axes[a][f] for f in cfg.TOP3 for a in range(3)], axis=1),
}
print(f"{len(windows)} windows, {len(np.unique(users))} users; "
      + ", ".join(f"{k}: {v.shape[1]} features" for k, v in FEATURES.items()))

10367 windows, 36 users; 40 statistical: 40 features, 33 temporal: 33 features, 3 FIN features: 9 features


## Experiment

In [7]:
MODELS = {
    "Random forest": lambda seed: RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
    "XGBoost": lambda seed: XGBClassifier(n_estimators=100, random_state=seed, eval_metric="mlogloss"),
    "Logistic regression": lambda seed: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=seed)),
    "1-NN": lambda seed: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1, p=1)),
    "SVM": lambda seed: make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}

rows = []
for seed in SEEDS:
    picked = np.random.default_rng(seed).choice(candidates, N_TRAIN + N_VAL, replace=False)
    tr = np.isin(users, picked[:N_TRAIN])
    te = ~np.isin(users, picked)  # the 2 validation users are left out
    for model, make in MODELS.items():
        for features, x in FEATURES.items():
            pred = make(seed).fit(x[tr], labels[tr]).predict(x[te])
            rows.append({"seed": seed, "model": f"{model} ({features})",
                         "accuracy": np.mean(pred == labels[te]),
                         "macro F1": f1_score(labels[te], pred, average="macro")})
    print(f"seed {seed}: train {sorted(picked[:N_TRAIN].tolist())}, test {te.sum()} windows done")
results = pd.DataFrame(rows)

seed 0: train [3, 7, 9, 10, 15, 18, 23], test 7978 windows done
seed 1: train [2, 5, 9, 13, 15, 29, 34], test 8032 windows done
seed 2: train [4, 8, 9, 12, 13, 23, 35], test 8060 windows done
seed 3: train [2, 3, 8, 20, 26, 29, 32], test 7382 windows done
seed 4: train [3, 16, 20, 27, 31, 34, 36], test 7324 windows done


In [8]:
order = list(dict.fromkeys(results["model"]))
g = results.groupby("model")[["accuracy", "macro F1"]]
display((g.mean().round(3).astype(str) + " Ã‚Â± " + g.std().round(3).astype(str)).loc[order],
        results.pivot(index="seed", columns="model", values="accuracy")[order].round(3))

,accuracy,macro F1
model,,
Random forest (40 statistical),0.762 Ã‚Â± 0.043,0.716 Ã‚Â± 0.028
Random forest (33 temporal),0.872 Ã‚Â± 0.015,0.787 Ã‚Â± 0.025
Random forest (3 FIN features),0.57 Ã‚Â± 0.023,0.471 Ã‚Â± 0.023
XGBoost (40 statistical),0.739 Ã‚Â± 0.044,0.686 Ã‚Â± 0.027
XGBoost (33 temporal),0.861 Ã‚Â± 0.028,0.776 Ã‚Â± 0.045
XGBoost (3 FIN features),0.569 Ã‚Â± 0.028,0.469 Ã‚Â± 0.025
Logistic regression (40 statistical),0.751 Ã‚Â± 0.029,0.694 Ã‚Â± 0.016
Logistic regression (33 temporal),0.816 Ã‚Â± 0.021,0.714 Ã‚Â± 0.027
Logistic regression (3 FIN features),0.584 Ã‚Â± 0.022,0.473 Ã‚Â± 0.028


model,Random forest (40 statistical),Random forest (33 temporal),Random forest (3 FIN features),XGBoost (40 statistical),XGBoost (33 temporal),XGBoost (3 FIN features),Logistic regression (40 statistical),Logistic regression (33 temporal),Logistic regression (3 FIN features),1-NN (40 statistical),1-NN (33 temporal),1-NN (3 FIN features),SVM (40 statistical),SVM (33 temporal),SVM (3 FIN features)
seed,,,,,,,,,,,,,,,
0,0.767,0.859,0.553,0.731,0.854,0.554,0.737,0.804,0.600,0.669,0.764,0.509,0.746,0.780,0.556
1,0.814,0.879,0.579,0.787,0.883,0.581,0.796,0.841,0.592,0.756,0.833,0.535,0.768,0.847,0.596
2,0.777,0.883,0.544,0.759,0.871,0.533,0.761,0.797,0.561,0.709,0.794,0.511,0.763,0.783,0.574
3,0.695,0.854,0.574,0.669,0.816,0.568,0.722,0.802,0.560,0.709,0.807,0.557,0.740,0.820,0.579
4,0.757,0.886,0.601,0.751,0.882,0.607,0.738,0.835,0.606,0.723,0.820,0.572,0.764,0.844,0.611
